# Case ascertainment of rare disease phenotypes in Our Future Health

## Purpose
This notebook extracts and harmonises disease case information for selected rare disease phenotypes in the Our Future Health (OFH) cohort using linked hospital inpatient data.

## Outputs
- Intermediate metadata and SQL query files used to extract diagnosis fields from linked datasets.
- Aggregated case counts for selected phenotypes derived from:
  - NHS England admitted patient care (HES inpatient) records

## Relationship to manuscript
Outputs from this notebook are used to generate **Extended Data Figures 6-7** in the main text and to populate **Supplementary Table 15** (*Prevalence and demographics obtained for rare diseases in Our Future Health and UK Biobank*).

## Data and access notes
Analyses use restricted Our Future Health data accessed within the OFH Trusted Research Environment under approved study permissions. All outputs are aggregated, non-disclosive summary statistics and comply with OFH Safe Output requirements.

## Notes
ICD-10 codes are taken from: Patrick, M. T., Bardhi, R., Zhou, W., Elder, J. T., Gudjonsson, J. E., & Tsoi, L. C. (2022). Enhanced rare disease mapping for phenome-wide genetic association in the UK Biobank. Genome Medicine, 14(1), 85.


In [ ]:
# Download ofh_tools
!dx download profile_study_v4:/applets/ofh_tools -r

## Setup env

# Import packages
import dxpy
import shlex
import subprocess
import numpy as np
import pandas as pd
import pyspark
from pyspark.sql import SparkSession

# Import ofh_tools
import ofh_tools

### Initialize Spark

spark = SparkSession.builder \
    .appName("Phenotype Analysis") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.kryoserializer.buffer.max", "128") \
    .getOrCreate()

### Load fields

files = [
    "table_s2_nhse_inpat_diag_icd_fields.csv",
    "rare_disease_phenotypes.csv",
    "imd_nhse_inpat.csv"
]

ofh_tools.utils.download_files([
    (str(ofh_tools.utils.find_latest_dx_file_id(f)), f"inputs/{f}")
    for f in files
])

pheno_dfs = {f.replace('.csv', ''): pd.read_csv(f'./inputs/{f}') for f in files}

metadata_dfs = ofh_tools.load.metadata()

## Rare diseases

ofh_tools.load.field_list(
    input_file="inputs/table_s2_nhse_inpat_diag_icd_fields.csv", 
    output_file="outputs/intermediate/nhse_inpat_diag_fields_metadata.csv",
)

ofh_tools.extract.fields(
    input_file="outputs/intermediate/nhse_inpat_diag_fields_metadata.csv",
    output_file="outputs/raw/nhse_inpat_diag_fields_raw_values_query.sql", 
    cohort_key="FULL_SAMPLE_ID", 
    sql_only=True
)

raw_nhse_inpat_df = ofh_tools.extract.sql_to_pandas(
    "outputs/raw/nhse_inpat_diag_fields_raw_values_query.sql"
)

trait_to_pids, trait_counts = ofh_tools.icd.match_icd_traits(
    raw_df=raw_nhse_inpat_df,
    traits_and_codes=pheno_dfs['rare_disease_phenotypes'][["trait", "ICD_code"]],
    pid_col="nhse_eng_inpat.pid",
    prefix_if_len_at_most=4,          # 'E101' -> prefix match
    return_occurrence_counts=True,
    use_pyarrow_strings=True,         # faster & smaller if pandas>=2.0 + pyarrow installed
    chunksize=500_000                 # tune: 250k–1M depending on instance type
)

# Strict case definition (primary diagnosis field only)
trait_counts

rare_df = trait_counts.sort_values(by='trait', ascending=True)

rare_df.to_csv('outputs/rare_disease_prevalence.csv')

##### Get sex-specific diagnoses

- Update parameters below to obtain results for TableS15

trait_to_pids, trait_counts = ofh_tools.icd.match_icd_traits(
    raw_df=raw_nhse_inpat_df.loc[(raw_nhse_inpat_df['participant.demog_sex_1_1'] == 1.0) | # 2.0 = female, 1.0 = male
                                 (raw_nhse_inpat_df['participant.demog_sex_2_1'] == 1.0)],
    traits_and_codes=pheno_dfs['rare_disease_phenotypes'][["trait", "ICD_code"]],
    pid_col="nhse_eng_inpat.pid",
    prefix_if_len_at_most=4,          # 'E101' -> prefix match
    return_occurrence_counts=True,
    primary_only=False,    # restrict to diag_4_1
    use_pyarrow_strings=True,         # faster & smaller if pandas>=2.0 + pyarrow installed
    chunksize=500_000                 # tune: 250k–1M depending on RAM
)


rare_df_m = trait_counts.sort_values(by='trait', ascending=True)

rare_df_m